# Project-R Simulator — EDA + GPU/TPU-accelerated indicator batch

Reads the real `data/project-r.db` SQLite file (oi_intraday, fyers_candles, bhavcopy_days, trade_suggestions, fno_stocks) and:
1. Audits data coverage per table/date (there's a real gap here — see Section 1).
2. Explores R-Factor / OI-level / sector-breadth distributions.
3. Batch-computes ATR(14) + session VWAP for **all** F&O symbols in one vectorized `torch` pass on whatever accelerator this runtime has (GPU/TPU/CPU), and cross-checks the result against the already-validated TypeScript implementation (`lib/signals/indicators.ts`, proven via `scripts/validate-indicators.ts`).

**To use a GPU or TPU:**
- **Colab**: Runtime → Change runtime type → Hardware accelerator → GPU (or TPU). TPU also needs `torch_xla` — see the commented install line in the next cell.
- **Kaggle**: Notebook settings (right sidebar) → Accelerator → GPU T4 x2.

The `data/project-r.db` file itself must come from `D:\Learnings\Project-R\Project-R-simulator\data\project-r.db` — the Colab-detection cell below prompts an upload if it doesn't find the file locally.

**Honesty note on GPU benefit**: the current universe is 182 symbols × 75 five-minute bars for one session (~13.6k rows) — small enough that GPU vs CPU won't show a dramatic wall-clock difference here. The value of this section is (a) the batched computation is *correct* (cross-checked below) and (b) the same code scales unmodified if the universe grows (e.g. NIFTY 500 F&O × many recorded days for a future backtest engine).

In [ ]:
# Colab / Kaggle ship pandas, matplotlib, torch preinstalled. If running a bare
# local kernel instead, uncomment:
# !pip install pandas matplotlib torch
#
# TPU support in Colab needs one extra package (skip if you only want GPU/CPU):
# !pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html -q

import json
import os
import sqlite3
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

In [ ]:
# ── Hardware detection — real, not decorative. Prints exactly what this
# runtime gave us; every tensor op below runs on `device`.
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_xla.core.xla_model as xm

        device = xm.xla_device()
        print(f"TPU detected: {device}")
    except ImportError:
        print(
            "No GPU/TPU detected — running on CPU.\n"
            "Colab: Runtime > Change runtime type > GPU or TPU.\n"
            "Kaggle: Notebook settings > Accelerator > GPU T4 x2."
        )
print(f"Using device: {device}")

In [ ]:
# ── DB connection — local path when run with a local kernel; upload prompt
# when the kernel is a real remote Colab/Kaggle runtime with no local filesystem access to this repo.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files

    print("Running in Colab — upload data/project-r.db from your local repo:")
    uploaded = files.upload()
    DB_PATH = next(iter(uploaded.keys()))
else:
    DB_PATH = "../data/project-r.db"

assert os.path.exists(DB_PATH), f"DB not found at {DB_PATH}"
conn = sqlite3.connect(DB_PATH)
print(f"Connected: {DB_PATH}")

## Section 1 — Data coverage audit

`oi_intraday` (an older, independent recorder) and `fyers_candles` (the 5-min OHLCV+OI recorder, live since 2026-07-03) don't fully overlap. This matters for anything that replays past sessions — e.g. `scripts/replay-lib.ts` needs `fyers_candles` for opening-range/ATR/Supertrend/SL, so days with `oi_intraday` but no `fyers_candles` silently produce zero picks (a data-coverage artifact, not a real gate-tightness signal).

In [ ]:
oi_dates = pd.read_sql_query("SELECT DISTINCT date FROM oi_intraday ORDER BY date", conn)["date"].tolist()
fy_dates = pd.read_sql_query("SELECT DISTINCT date FROM fyers_candles ORDER BY date", conn)["date"].tolist()
bhav_dates = pd.read_sql_query("SELECT DISTINCT date FROM bhavcopy_days ORDER BY date", conn)["date"].tolist()

print(f"oi_intraday: {len(oi_dates)} dates -> {oi_dates}")
print(f"fyers_candles: {len(fy_dates)} dates -> {fy_dates}")
print(f"bhavcopy_days: {len(bhav_dates)} dates, range {bhav_dates[0]} .. {bhav_dates[-1]}")

coverage = pd.DataFrame({"date": oi_dates})
coverage["has_fyers_candles"] = coverage["date"].isin(fy_dates)
coverage["has_bhavcopy"] = coverage["date"].isin(bhav_dates)
coverage

In [ ]:
rows_per_date = pd.read_sql_query("SELECT date, COUNT(*) AS oi_rows FROM oi_intraday GROUP BY date", conn)
fy_rows_per_date = pd.read_sql_query(
    "SELECT date, COUNT(*) AS fyers_rows FROM fyers_candles GROUP BY date", conn
)
merged = rows_per_date.merge(fy_rows_per_date, on="date", how="left").fillna(0)
merged.plot(x="date", y=["oi_rows", "fyers_rows"], kind="bar", figsize=(10, 4), title="Row coverage per date")
plt.tight_layout()
plt.show()

## Section 2 — R-Factor / OI-level distributions

In [ ]:
rf = pd.read_sql_query(
    "SELECT date, symbol, oiLevel, futTurnover, changePctOpen, spreadPct, imbalance FROM oi_intraday", conn
)
display(rf.groupby("date")["oiLevel"].describe())

rf["oiLevel"].hist(bins=40, figsize=(8, 4))
plt.title("Distribution of intraday OI level (fut OI \u00f7 20d avg) across all recorded sessions")
plt.xlabel("OI level (\u00d7)")
plt.show()

## Section 3 — `/trade-suggest` picks so far

Only 3 rows exist right now — too few for statistics, but shown in full since every row is real.

In [ ]:
sug = pd.read_sql_query("SELECT * FROM trade_suggestions ORDER BY date, rank", conn)
display(sug[["date", "symbol", "optionType", "rank", "score", "spotAtSuggest", "maxUpPct", "maxDownPct", "closePct"]])

for _, row in sug.iterrows():
    print(f"{row['date']} {row['symbol']} {row['optionType']} rank {row['rank']} score {row['score']:.3f}")
    try:
        for reason in json.loads(row["reasons"]):
            print(f"    - {reason}")
    except Exception:
        pass
    print()

## Section 4 — Sector breadth (latest bhavcopy session)

In [ ]:
bhav = pd.read_sql_query(
    """SELECT b.date, b.symbol, f.sector, b.eqClose
       FROM bhavcopy_days b JOIN fno_stocks f ON b.symbol = f.symbol
       WHERE f.isIndex = 0 ORDER BY b.symbol, b.date""",
    conn,
)
bhav["pctChange"] = bhav.groupby("symbol")["eqClose"].pct_change() * 100
latest_date = bhav["date"].max()
latest = bhav[bhav["date"] == latest_date]
sector_breadth = latest.groupby("sector")["pctChange"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(f"Latest bhavcopy session: {latest_date}")
display(sector_breadth)

## Section 5 — GPU/TPU-accelerated indicator batch (ATR14 + session VWAP, all symbols at once)

Loads every symbol's recorded 5-min EQ bars for 2026-07-03, stacks them into `(bars, symbols)` tensors, and computes True Range → Wilder ATR(14) and session VWAP for **all symbols simultaneously** on `device`. The ATR recursion is inherently sequential over time (Wilder smoothing depends on the prior value), but every time-step updates all symbols in parallel — this is the shape of computation that scales with a bigger symbol universe on a GPU/TPU, unlike the current one-symbol-at-a-time TypeScript loop in `lib/signals/indicators.ts`.

The math is written to match that TypeScript implementation exactly (same True Range formula, same seed-bar convention, same Wilder smoothing), and the next cell cross-checks 5 symbols against the values `scripts/validate-indicators.ts` already printed and verified earlier — RELIANCE, DMART, HDFCBANK, POWERINDIA, MARICO.

In [ ]:
eq = pd.read_sql_query(
    """SELECT symbol, bucketTs, open, high, low, close, volume
       FROM fyers_candles WHERE date = '2026-07-03' AND instrument = 'EQ'
       ORDER BY symbol, bucketTs""",
    conn,
)
bar_counts = eq.groupby("symbol").size()
max_bars = bar_counts.max()
full_bar_symbols = sorted(bar_counts[bar_counts == max_bars].index.tolist())
print(f"{len(full_bar_symbols)} of {eq['symbol'].nunique()} symbols have the full {max_bars} bars — using those")

eq_full = eq[eq["symbol"].isin(full_bar_symbols)]


def pivot_field(field):
    p = eq_full.pivot(index="bucketTs", columns="symbol", values=field)
    return p[full_bar_symbols].values


t0 = time.time()
H = torch.tensor(pivot_field("high"), dtype=torch.float64, device=device)
L = torch.tensor(pivot_field("low"), dtype=torch.float64, device=device)
C = torch.tensor(pivot_field("close"), dtype=torch.float64, device=device)
V = torch.tensor(pivot_field("volume"), dtype=torch.float64, device=device)

# True range, vectorized across ALL symbols — matches lib/signals/indicators.ts trueRanges()
prevC = torch.cat([C[:1], C[:-1]], dim=0)
tr_full = torch.maximum(H - L, torch.maximum((H - prevC).abs(), (L - prevC).abs()))
tr_full[0] = H[0] - L[0]  # seed-bar convention matches the TS trueRanges()
trs_used = tr_full[1:]  # TS drops the seed bar via .slice(1) before the Wilder average

PERIOD = 14
atr = trs_used[:PERIOD].mean(dim=0)
for i in range(PERIOD, trs_used.shape[0]):
    atr = (atr * (PERIOD - 1) + trs_used[i]) / PERIOD  # Wilder smoothing — sequential over time, parallel over symbols

# Session VWAP — matches lib/signals/indicators.ts sessionVwap(), fully vectorized (no loop needed)
typical = (H + L + C) / 3
vwap = (typical * V).sum(dim=0) / V.sum(dim=0)

if device.type == "cuda":
    torch.cuda.synchronize()
elapsed_ms = (time.time() - t0) * 1000
print(f"Batched ATR14 + VWAP over {len(full_bar_symbols)} symbols x {max_bars} bars on {device}: {elapsed_ms:.2f} ms")

results = pd.DataFrame({"symbol": full_bar_symbols, "atr14": atr.cpu().numpy(), "vwap": vwap.cpu().numpy()})
results = results.sort_values("atr14", ascending=False).reset_index(drop=True)
display(results.head(15))

In [ ]:
# Cross-check vs the REAL values scripts/validate-indicators.ts printed and
# validated earlier this session — proves this batched torch path is correct,
# not just fast.
checks = {
    "RELIANCE": (1.22, 1305.12),
    "DMART": (4.74, 4004.58),
    "HDFCBANK": (0.74, 802.81),
    "POWERINDIA": (120.27, 31204.04),
    "MARICO": (1.80, 846.55),
}
print("Cross-check vs TS-validated (scripts/validate-indicators.ts):")
all_ok = True
for sym, (exp_atr, exp_vwap) in checks.items():
    row = results[results["symbol"] == sym]
    if row.empty:
        print(f"  {sym}: not in full-bar set — skip")
        continue
    got_atr = row["atr14"].values[0]
    got_vwap = row["vwap"].values[0]
    atr_ok = abs(got_atr - exp_atr) < 0.01
    vwap_ok = abs(got_vwap - exp_vwap) < 0.01
    all_ok = all_ok and atr_ok and vwap_ok
    print(
        f"  {sym}: ATR14 {got_atr:.2f} vs {exp_atr} {'OK' if atr_ok else 'MISMATCH'} | "
        f"VWAP {got_vwap:.2f} vs {exp_vwap} {'OK' if vwap_ok else 'MISMATCH'}"
    )
print("ALL CROSS-CHECKS PASSED" if all_ok else "MISMATCH — investigate before trusting this device's output")

## Notes

- This section's logic was verified locally on CPU torch (no GPU available in the dev sandbox) and matched the TypeScript reference exactly on all 5 spot-checked symbols before this notebook was written — see the run log in the conversation that produced this file.
- The same code runs unmodified on CUDA (`device.type == 'cuda'`) or TPU (`torch_xla`) — only the `device` object changes.
- At 182 symbols \u00d7 75 bars, don't expect a dramatic GPU speedup over CPU — the win shows up once the universe or history grows (e.g. NIFTY 500 \u00d7 many sessions for a future backtest engine).